<a href="https://colab.research.google.com/github/lydiako861-a11y/Rust-conda/blob/main/Telegram_Command_Controller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import asyncio
import redis.asyncio as redis
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import Application, CommandHandler, CallbackQueryHandler, ContextTypes

# Configuration from Environment
TOKEN = os.getenv("8778574987:AAFMWxdYvFlJ1wCvIhx5qfK7F0mZGmLnBIw")
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
ADMIN_CHAT_ID = os.getenv("ADMIN_CHAT_ID")

class AetherController:
    def __init__(self):
        self.r = redis.from_url(REDIS_URL, decode_responses=True)

    async def start(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """Welcome message and status check"""
        await update.message.reply_text(
            "🛰 **Aether-Link Command Center**\n"
            "Status: Agents Online\n\n"
            "Use /status to see current signals or /halt for emergency shutdown."
        )

    async def status(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """Fetch current signals from Redis Priority Queue"""
        signals = await self.r.zrange("arbitrage:signals", 0, 2, withscores=True)
        if not signals:
            await update.message.reply_text("No active signals in queue.")
            return

        for sig_json, score in signals:
            sig = json.loads(sig_json)
            text = (
                f"🔥 **Signal Detected** (Spread: {sig['opportunity']['expected_spread_bps']}bps)\n"
                f"Pair: {sig['pair']['base']}/{sig['pair']['quote']}\n"
                f"Venues: {sig['opportunity']['buy_venue']} -> {sig['opportunity']['sell_venue']}\n"
                f"Risk Score: {sig['risk_score']}"
            )
            keyboard = [[
                InlineKeyboardButton("✅ Execute Now", callback_data=f"exec_{sig['signal_id']}"),
                InlineKeyboardButton("❌ Dismiss", callback_data=f"drop_{sig['signal_id']}")
            ]]
            reply_markup = InlineKeyboardMarkup(keyboard)
            await update.message.reply_text(text, reply_markup=reply_markup, parse_mode='Markdown')

    async def handle_action(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """Handle button clicks for manual execution"""
        query = update.callback_query
        await query.answer()

        action, sig_id = query.data.split('_')
        if action == "exec":
            # Push to the 'execution:priority' queue for Agent 2
            await self.r.lpush("execution:priority", sig_id)
            await query.edit_message_text(text=f"🚀 Manual Execution Triggered for {sig_id}")
        else:
            await query.edit_message_text(text="🗑 Signal dismissed.")

    async def halt(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """Emergency Halt - Triggers MCP emergency_halt tool"""
        await self.r.set("system:halt", "true")
        await update.message.reply_text("🛑 **EMERGENCY HALT ACTIVATED**\nAll agent activity suspended.")

if __name__ == "__main__":
    controller = AetherController()
    app = Application.builder().token(TOKEN).build()

    app.add_handler(CommandHandler("start", controller.start))
    app.add_handler(CommandHandler("status", controller.status))
    app.add_handler(CommandHandler("halt", controller.halt))
    app.add_handler(CallbackQueryHandler(controller.handle_action))

    print("Aether-Link Controller started...")
    app.run_polling()